# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# The Croissant schema URL for FAIR² dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This will help us select which data to work with in detail.

In [ ]:
# Display the available record sets, fields, columns and their @ids
print("Available record sets:\n")
if hasattr(metadata, "record_sets") and len(metadata.record_sets) > 0:
    for rs in metadata.record_sets:
        print(f"- RecordSet @id: {rs.id}")
        print(f"  Name: {getattr(rs, 'name', '')}")
        print(f"  Description: {getattr(rs, 'description', '')}")
        if hasattr(rs, "fields"):
            print("  Fields:")
            for field in rs.fields:
                print(f"    - Field @id: {field.id}, Name: {getattr(field, 'name', '')}, DataType: {getattr(field, 'data_type', '')}")
        print()
else:
    print("No record sets are defined in the metadata.")

For this dataset, the record sets may not be directly listed in metadata's `record_sets` (sometimes Croissant puts everything in one main table/record set). Let's enumerate what is possible to load using the mlcroissant API. If no explicit record sets exist, use the default provided by the dataset.

In [ ]:
# Find all available record set @ids
record_set_ids = list(dataset.record_set_ids)
print("Discovered record set @ids:")
for rid in record_set_ids:
    print(rid)

# For each record set, enumerate the field @ids
for record_set_id in record_set_ids:
    print(f"\nFields for RecordSet @id: {record_set_id}")
    for field in dataset.field_ids(record_set=record_set_id):
        print(f"- Field @id: {field}")

## 3. Data Extraction
Let's load the main record set into a pandas DataFrame. We'll use the record set and field `@id`s from the overview above. If only one record set is available, we'll use that for demonstration.

In [ ]:
# For this dataset, we'll use all discovered record sets (likely just one)
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows. Available columns (@id):\n{df.columns.tolist()}")

# For demonstration, choose the first record set for further analysis
main_record_set_id = record_set_ids[0]
print(f"\nPreview of data for RecordSet @id: {main_record_set_id}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We will:
- Select a numeric field for analysis (e.g., age at diagnosis, interval, or similar).
- Filter records based on a threshold for this field.
- Normalize its values.
- Group records by a categorical/group field (e.g., sex, MSI-H status, anatomical location).

All fields will be referenced by their `@id` as per the Croissant schema.

In [ ]:
# List columns to help user pick a numeric and a group (categorical) field
main_df = dataframes[main_record_set_id]
print("Columns available (@id):")
print(main_df.columns.tolist())

# For demonstration, assign field @ids to variables.
# You can replace these with any valid numeric and grouping field @ids discovered above.
# Example guesses based on possible clinical variables:
numeric_field_id = None
group_field_id = None
# Try to guess common field names
for col in main_df.columns:
    if "age" in col.lower():
        numeric_field_id = col
    if ("sex" in col.lower() or "gender" in col.lower()):
        group_field_id = col
if not numeric_field_id:
    # Try to use diagnosis interval or any field with 'interval' or 'years' in name
    for col in main_df.columns:
        if "interval" in col.lower() or "year" in col.lower():
            numeric_field_id = col
            break
if not numeric_field_id:
    # Otherwise pick the first float/int-looking column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break
if not group_field_id:
    # Try MSI status or location
    for col in main_df.columns:
        if ("msi" in col.lower() or "status" in col.lower() or "loc" in col.lower()):
            group_field_id = col
            break

if numeric_field_id and group_field_id:
    print(f"Numeric field selected (@id): {numeric_field_id}")
    print(f"Grouping field selected (@id): {group_field_id}")
else:
    print("Could not auto-detect numeric/grouping field. Please adjust variable assignments above if needed.")

# Clean numeric field (convert to float, coerce errors)
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

# Filtering: Choose a threshold, e.g., mean, or a static cutoff (e.g., age > 50)
threshold = main_df[numeric_field_id].mean() if pd.notnull(main_df[numeric_field_id].mean()) else 0
filtered_df = main_df[main_df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize
filtered_df.loc[:, f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping and aggregate
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    grouped_df = grouped_df.rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships between fields. All field references should use their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7, 4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group (if enough groups and non-null values)
if group_field_id and group_field_id in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30, ha='right')
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded a clinical dataset from a Croissant schema using `mlcroissant`.
- Explored its metadata, record sets, fields, and columns using their `@id` identifiers.
- Loaded the main table into a DataFrame and performed simple statistics and cleaning.
- Conducted exploratory filtering, normalization, grouping, and generated visualizations.

By referencing all data elements using `@id`, your analysis remains robust and consistent for reproducible and FAIR data science.
